# TP Kafka : meteo en temps reel

## Exercice 1 : topic et producteur simple

1. Creer le topic `weather_stream`.
2. Envoyer un message statique `{"msg": "Hello Kafka"}`.

In [4]:
from kafka.admin import KafkaAdminClient, NewTopic

admin = KafkaAdminClient(bootstrap_servers="kafka:9092")
if "weather_stream" not in admin.list_topics():
    admin.create_topics([NewTopic("weather_stream", 1, 1)])
print(admin.list_topics())

['weather_stream']


In [5]:
from kafka import KafkaProducer

producer = KafkaProducer(bootstrap_servers="kafka:9092")

meta = producer.send("weather_stream", b'{"msg": "Hello Kafka"}').get(timeout=10)
print(f"envoye -> partition {meta.partition}, offset {meta.offset}")

envoye -> partition 0, offset 2


## Exercice 3 : streaming Open-Meteo

`exo3.py` interroge l'API Open-Meteo toutes les 30 s pour les 5 villes et envoie chaque reponse + `city`/`country` dans `weather_stream`. Daemon : `docker exec -d pyspark_notebook python work/exo3.py`

Preuve : `exo2.py` lit le flux en direct — les 5 villes defilent avec `temperature_2m`, `wind_speed_10m`, `weather_code`.

![messages meteo dans weather_stream](exo3.png)

## Exercice 4 : transformation Spark et alertes

Lit `weather_stream` en continu, calcule `wind_alert_level` et `heat_alert_level`, ecrit dans `weather_transformed`.

- vent : < 10 m/s `level_0` - 10-20 `level_1` - > 20 `level_2`
- chaleur : < 25 C `level_0` - 25-35 `level_1` - > 35 `level_2`
- `event_time` = timestamp d'ecriture du message dans Kafka (colonne `timestamp` de la source)

In [9]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T

spark = (SparkSession.builder
         .appName("weather-transform")
         .master("spark://spark-master:7077")
         .getOrCreate())

schema = T.StructType([
    T.StructField("city", T.StringType()),
    T.StructField("country", T.StringType()),
    T.StructField("current", T.StructType([
        T.StructField("temperature_2m", T.DoubleType()),
        T.StructField("wind_speed_10m", T.DoubleType()),
        T.StructField("weather_code", T.IntegerType()),
    ])),
])

brut = (spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", "kafka:9092")
    .option("subscribe", "weather_stream")
    .option("startingOffsets", "earliest")
    .load())

In [10]:
meteo = (brut
    .select(F.from_json(F.col("value").cast("string"), schema).alias("d"),
            F.col("timestamp").alias("event_time"))
    .select("event_time", "d.*")
    .filter(F.col("city").isNotNull())   # ecarte les messages hors schema (Hello Kafka)
    .select("event_time",
            F.col("current.temperature_2m").alias("temperature"),
            F.col("current.wind_speed_10m").alias("windspeed"),
            "city", "country",
            F.col("current.weather_code").alias("weather_code"))
    .withColumn("wind_alert_level",
        F.when(F.col("windspeed") < 10, "level_0")
         .when(F.col("windspeed") <= 20, "level_1")
         .otherwise("level_2"))
    .withColumn("heat_alert_level",
        F.when(F.col("temperature") < 25, "level_0")
         .when(F.col("temperature") <= 35, "level_1")
         .otherwise("level_2")))

requete = (meteo.select(F.to_json(F.struct("*")).alias("value"))
    .writeStream.format("kafka")
    .option("kafka.bootstrap.servers", "kafka:9092")
    .option("topic", "weather_transformed")
    .option("checkpointLocation", "/home/jovyan/work/checkpoints/exo4")
    .outputMode("append")
    .start())
print(requete.status)

{'message': 'Initializing sources', 'isDataAvailable': False, 'isTriggerActive': False}


Verification : on relit `weather_transformed` comme un consommateur classique (5 messages).

In [11]:
import json
from kafka import KafkaConsumer

consumer = KafkaConsumer("weather_transformed", bootstrap_servers="kafka:9092",
    auto_offset_reset="earliest",
    value_deserializer=lambda b: json.loads(b.decode()),
    consumer_timeout_ms=60000)
for i, m in enumerate(consumer):
    print(m.value)
    if i >= 4: break

{'event_time': '2026-11-10T13:27:08.759Z', 'temperature': 12.1, 'windspeed': 2.08, 'city': 'Paris', 'country': 'France', 'weather_code': 80, 'wind_alert_level': 'level_0', 'heat_alert_level': 'level_0'}
{'event_time': '2026-11-10T13:27:08.935Z', 'temperature': 21.4, 'windspeed': 5.82, 'city': 'Seville', 'country': 'Spain', 'weather_code': 80, 'wind_alert_level': 'level_0', 'heat_alert_level': 'level_0'}
{'event_time': '2026-11-10T13:27:09.118Z', 'temperature': 30.6, 'windspeed': 3.62, 'city': 'Dubai', 'country': 'United Arab Emirates', 'weather_code': 0, 'wind_alert_level': 'level_0', 'heat_alert_level': 'level_1'}
{'event_time': '2026-11-10T13:27:09.297Z', 'temperature': 6.7, 'windspeed': 6.8, 'city': 'Reykjavik', 'country': 'Iceland', 'weather_code': 3, 'wind_alert_level': 'level_0', 'heat_alert_level': 'level_0'}
{'event_time': '2026-11-10T13:27:09.488Z', 'temperature': 15.5, 'windspeed': 7.44, 'city': 'Wellington', 'country': 'New Zealand', 'weather_code': 51, 'wind_alert_level': '